# Geography chatbot

In Colab, open **Secrets** (the key icon), add `OPENAI_API_KEY`, and enable notebook access. Run the cells in order. Ask a geography question or type `stop` to end the chat.

In [ ]:
%pip install -q langgraph langchain langchain-openai

In [ ]:
import os
from typing import Annotated, TypedDict

from google.colab import userdata
from langchain.chat_models import init_chat_model
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
model = init_chat_model("openai:gpt-6-luna")

SYSTEM = SystemMessage(
    content="You are a geography tutor for students. Answer only geography "
    "questions, including follow-ups. Politely decline other topics and invite "
    "the student to ask a geography question."
)


class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


def chatbot(state: ChatState) -> ChatState:
    reply = model.invoke([SYSTEM] + state["messages"])
    return {"messages": [reply]}


builder = StateGraph(ChatState)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)
graph = builder.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "1"}}

In [ ]:
while True:
    text = input("You: ").strip()
    if text.lower() == "stop":
        break

    result = graph.invoke({"messages": [HumanMessage(text)]}, config)
    print("Geography bot:", result["messages"][-1].text)